# Chapter 15 &mdash; Worked Mapping Reductions

**Concept 8 of the Chapter 15 decomposition:** *Worked Mapping Reductions: $A_{TM}\leq_m Halt_{TM}$ and $A_{TM}\leq_m \overline{E_{TM}}$*

$A_{TM}\le_m Halt_{TM}$ and $A_{TM}\le_m \overline{E_{TM}}$ &mdash; $f$ writes out the <i>text</i> of a new machine.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter15/Concept-Worked-Mapping-Reductions/Concept-Worked-Mapping-Reductions.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Run this cell first. It works both on Colab and on your own machine.
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
import sys

try:                       # -- are we on Colab? --
    import google.colab
    OWN_INSTALL = False
except ImportError:
    OWN_INSTALL = True

if OWN_INSTALL:
    # Running from Jove/Chapter<N>/Concept-<Name>/ : reach the Jove root.
    sys.path[0:0] = ['../..', '../../3rdparty',
                     '../../..', '../../../3rdparty',
                     '..', '../3rdparty', '.']
else:
    ! if [ ! -d Jove ]; then git clone -q https://github.com/ganeshutah/Jove Jove; fi
    sys.path.append('./Jove')
    sys.path.append('./Jove/jove')

# -- imports needed by this notebook --
from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.Def_NFA        import *
from jove.LangDef        import *
from jove.Def_TM         import *
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
print("Jove loaded. Ready.")

## 1. The idea


Two reductions in full, both with the same shape: **$f$ writes out the source code of
a new machine.** It never runs anything.

**$A_{TM} \le_m Halt_{TM}$.** $f(\langle M,w\rangle) = \langle M',w\rangle$ where
$M'$ runs $M$ on its input, **halts** if $M$ accepts, and **loops** if $M$ rejects.
Then $M'$ halts on $w$ iff $M$ accepts $w$.

**$A_{TM} \le_m \overline{E_{TM}}$**, where $E_{TM}=\{\langle M\rangle : L(M)=\emptyset\}$.
$f(\langle M,w\rangle) = \langle M_w\rangle$ where $M_w$ **ignores its input** and runs
$M$ on the fixed string $w$. Then $L(M_w)$ is $\Sigma^*$ if $M$ accepts $w$ and
$\emptyset$ otherwise &mdash; so $M$ accepts $w$ iff $\langle M_w\rangle \notin E_{TM}$.

The move to internalise: **$f$ is a program that prints a program.**

## 2. Definitions

### f as a source-code transformer

In [ ]:
def f_halt(M_src, w):
    # writes the TEXT of M': run M; halt if it accepts, loop if it rejects
    return ("def M_prime(x):\n"
            "    if (%s)(x):\n"
            "        return 'HALT'\n"
            "    while True:\n"
            "        pass\n" % M_src), w

def f_empty(M_src, w):
    # writes the TEXT of M_w: ignore the input, run M on the fixed w
    return ("def M_w(x):\n"
            "    return (%s)(%r)\n" % (M_src, w))

### Executing the generated text, to check the claims

In [ ]:
def build(src, name):
    g = {}
    exec(src, g)
    return g[name]

## 3. Tests

**$f$ writes text.** Here is the text it writes.

In [ ]:
src, w = f_halt("lambda x: x.startswith('1')", '101')
print(src)
print("second component (unchanged) :", repr(w))

$M'$ halts on $w$ exactly when $M$ accepts $w$.

In [ ]:
import signal
def halts_quickly(fn, arg, limit=20000):
    # a crude step budget, standing in for "does it halt"
    import itertools
    try:
        # our generated loops are literal `while True`, so guard by inspection
        return fn(arg) is not None
    except Exception:
        return False

for M_src, w in [("lambda x: x.startswith('1')", '101'),
                 ("lambda x: x.startswith('1')", '011')]:
    accepts = eval(M_src)(w)
    print("  M accepts %r ? %-6s  so M' should %s"
          % (w, accepts, "HALT" if accepts else "LOOP"))
    src, _ = f_halt(M_src, w)
    Mp = build(src, 'M_prime')
    if accepts:
        assert Mp(w) == 'HALT'
        print("     M' returned HALT, as predicted")
    else:
        print("     M' would loop -- not run here, for obvious reasons")

**$f$ never runs $M$.** It only does string surgery.

In [ ]:
Loop = "lambda x: (_ for _ in ()).throw(RuntimeError('would loop'))"
src = f_empty(Loop, '000')
print("f produced source for a machine built from a diverging M:")
print(src)
print("f itself returned normally -- it never called M.")
assert isinstance(src, str)

The second reduction: $M_w$ **ignores its input**.

In [ ]:
src = f_empty("lambda x: x.startswith('1')", '101')
Mw = build(src, 'M_w')
print(src)
for x in ['', '0', '1111', 'anything']:
    print("   M_w(%-10r) = %s" % (x, Mw(x)))
assert all(Mw(x) is True for x in ['', '0', '1111'])
print("\nM accepts '101', so L(M_w) = Sigma* -- certainly not empty.")

And when $M$ rejects $w$, $L(M_w)=\emptyset$.

In [ ]:
src = f_empty("lambda x: x.startswith('1')", '011')
Mw = build(src, 'M_w')
for x in ['', '0', '1111']:
    print("   M_w(%-8r) = %s" % (x, Mw(x)))
assert all(Mw(x) is False for x in ['', '0', '1111'])
print("\nL(M_w) = {} -- so <M_w> IS in E_TM, and M does not accept w.")
print("Hence  M accepts w  <=>  <M_w> not in E_TM  <=>  <M_w> in complement(E_TM).")

Both reductions are **total and computable**, which is the requirement.

In [ ]:
for M_src in ["lambda x: True", "lambda x: False", Loop]:
    s1, _ = f_halt(M_src, 'w')
    s2 = f_empty(M_src, 'w')
    print("  f_halt and f_empty both returned text for %-30s ok" % M_src[:30])
print("\nEvery input produces an output, and no input causes f to diverge.")

## 4. Exercises


1. Write $f$ for $A_{TM} \le_m \{\langle M\rangle : M \text{ accepts } \varepsilon\}$.
2. Why does $M_w$ have to ignore its input?
3. Is $E_{TM}$ RE? Is $\overline{E_{TM}}$?

In [ ]:
# Your work for the exercises above.